# 04 - Evaluation and Figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfefGuesmi/msdd-arab-youth-space-hackathon/blob/main/MSDD-Net/notebooks/04_Evaluation_and_Figures.ipynb)

Loads trained checkpoints, evaluates on the MARIDA test set, produces the per-class tables, the ablation bar chart, the class distribution chart, and the multi-model qualitative figures.

You can skip training if you download the checkpoints from Google Drive.

## 1. Setup

In [ ]:
import os
if not os.path.isdir('msdd-arab-youth-space-hackathon'):
    !git clone -b msddnet https://github.com/AfefGuesmi/msdd-arab-youth-space-hackathon.git

%cd msdd-arab-youth-space-hackathon/MSDD-Net
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

## 2. Download checkpoints (skip if you trained in this session)

In [ ]:
DRIVE_CKPT_FOLDER = ''  # e.g. '1AbCdEfGhIjKlMnOpQrStUvWxYz'

if DRIVE_CKPT_FOLDER and not os.path.isdir('trained_models/unet'):
    !pip install -q gdown
    !gdown --folder $DRIVE_CKPT_FOLDER -O trained_models
    !ls trained_models

## 3. Evaluate each model on the MARIDA test set

In [ ]:
# U-Net
!python evaluation_swin_unetv2.py \
  --model_type unet --backbone none --hidden_channels 16 \
  --model_path trained_models/unet/best_model.pth \
  --tta False

In [ ]:
# MSDD-Net baseline
!python evaluation_swin_unetv2.py \
  --model_type swin --variant baseline \
  --model_path trained_models/baseline/best_model.pth \
  --tta True

In [ ]:
# MSDD-Net full
!python evaluation_swin_unetv2.py \
  --model_type swin --variant full \
  --model_path trained_models/full/best_model.pth \
  --tta True

## 4. Aggregate results table

In [ ]:
import json, os
import pandas as pd

def load(path):
    if not os.path.isfile(path):
        return None
    with open(path) as f:
        return json.load(f)

runs = {
    'U-Net':          'logs/evaluating_unet_unet_results.json',
    'MSDD-Net base':  'logs/evaluating_swin_baseline_tta_results.json',
    'MSDD-Net full':  'logs/evaluating_swin_full_tta_results.json',
}

rows = []
for name, path in runs.items():
    r = load(path)
    if r is None:
        continue
    pc = r['per_class']
    rows.append({
        'model': name,
        'MD F1': pc['Marine Debris']['f1'],
        'MD IoU': pc['Marine Debris']['iou'],
        'macro F1': pc['Average']['f1'],
        'macro mIoU': pc['Average']['iou'],
    })

df = pd.DataFrame(rows)
df

## 5. Ablation bar chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

GROUPS   = ['1', '2', '3', '4', '5', '6']
MACRO_F1 = [61.9, 64.8, 69.0, 72.2, 72.3, 79.0]
MIOU     = [52.0, 53.7, 57.0, 60.5, 61.3, 69.0]
C_F1, C_IOU = '#1a6b8a', '#a4dad5'

x = np.arange(len(GROUPS)); w = 0.38
fig, ax = plt.subplots(figsize=(11.5, 6.5))
b1 = ax.bar(x - w/2, MACRO_F1, w, label='Macro F1 (%)', color=C_F1)
b2 = ax.bar(x + w/2, MIOU,     w, label='mIoU (%)',     color=C_IOU)
for b, v in zip(b1, MACRO_F1): ax.annotate(f'{v:g}', (b.get_x()+b.get_width()/2, b.get_height()), xytext=(0,3), textcoords='offset points', ha='center')
for b, v in zip(b2, MIOU):     ax.annotate(f'{v:g}', (b.get_x()+b.get_width()/2, b.get_height()), xytext=(0,3), textcoords='offset points', ha='center')
ax.set_xticks(x); ax.set_xticklabels(GROUPS)
ax.set_ylim(40, 85); ax.set_yticks(np.arange(40, 86, 5))
ax.yaxis.grid(True, color='#cccccc'); ax.set_axisbelow(True)
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False)
plt.tight_layout(); plt.savefig('ablation_bars.png', dpi=200, bbox_inches='tight'); plt.show()

## 6. Class distribution (log scale, reference figure)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

CLASS_SHARE = {1:0.015, 2:0.040, 3:0.070, 4:0.150, 5:0.250, 6:0.400,
               7:0.600, 8:0.600, 9:1.000, 10:1.500, 11:3.000,
               12:13.0, 13:15.0, 14:20.0, 15:40.0}
HIGHLIGHT = 6

ids = sorted(CLASS_SHARE.keys(), reverse=True)
vals = [CLASS_SHARE[i] for i in ids]
cols = ['#f4513a' if i == HIGHLIGHT else '#1f6e8c' for i in ids]

fig, ax = plt.subplots(figsize=(11.5, 7.5))
y = np.arange(len(ids))
ax.barh(y, vals, color=cols, height=0.72)
ax.set_yticks(y); ax.set_yticklabels([str(i) for i in ids]); ax.invert_yaxis()
ax.set_xscale('log'); ax.set_xlim(0.01, 100)
ax.set_xticks([0.01, 0.1, 1, 10, 100])
ax.set_xticklabels(['0.01%','0.1%','1%','10%','100%'])
ax.xaxis.grid(True, color='#d0d0d0'); ax.set_axisbelow(True)
for s in ('top','right','left'): ax.spines[s].set_visible(False)
ax.set_title('Share of annotated pixels per class (log scale)', pad=14)
plt.tight_layout(); plt.savefig('class_share.png', dpi=200, bbox_inches='tight'); plt.show()

## 7. Multi-model qualitative figures

Requires RF predictions from notebook 01 and one or more trained checkpoints. Update the `--models` list to include whichever models you have.

In [ ]:
!python plot_qualitative.py \
  --data_path data \
  --n_patches 6 \
  --out_dir ../qualitative_results/marida_comparison \
  --models \
    "RF SS+SI+GLCM=trained_models/rf_ss_si_glcm/predictions" \
    "U-Net=trained_models/unet/best_model.pth" \
    "MSDD-Net baseline=trained_models/baseline/best_model.pth"

## 8. Display the qualitative figures inline

In [ ]:
from IPython.display import Image, display
import os

base = '../qualitative_results/marida_comparison'
for fn in ('comparison_all_classes.png', 'comparison_debris_focus.png'):
    p = os.path.join(base, fn)
    if os.path.isfile(p):
        print(fn)
        display(Image(p))
    else:
        print('Missing:', p)

## 9. Save all figures to Google Drive

In [ ]:
SAVE = True
if SAVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, os
    dst = '/content/drive/MyDrive/MSDD_figures'
    os.makedirs(dst, exist_ok=True)
    for f in ('ablation_bars.png', 'class_share.png'):
        if os.path.isfile(f):
            shutil.copy2(f, dst)
    src = '../qualitative_results/marida_comparison'
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(dst, 'marida_comparison'), dirs_exist_ok=True)
    print('Figures saved to', dst)